# Day 19 — Cross-validation and robust validation practices

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Cross-validation fundamentals

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-019/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The 23 test students are set aside first, as on Days 17 and 18; everything today happens inside the other 127
students = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "PreviousScore"]
pool, test = train_test_split(students, test_size=0.15, random_state=42, stratify=students["Passed"])
X, y = pool[features], pool["Passed"].astype(int)
model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=1000))])

# One split, twenty times: the same model, the same 127 students, a different random 80/20 split each time
accuracies = []
for seed in range(20):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    accuracies.append(model.fit(X_tr, y_tr).score(X_val, y_val))
print("Validation students per split:", len(X_val))
print("Accuracy on 20 different splits:")
print(np.round(accuracies[:10], 2))
print(np.round(accuracies[10:], 2))
print(f"Lowest {min(accuracies):.3f} | highest {max(accuracies):.3f} | mean {np.mean(accuracies):.3f}")

Validation students per split: 26
Accuracy on 20 different splits:
[0.85 0.92 0.73 0.92 0.73 0.73 0.69 0.58 0.73 0.81]
[0.85 0.65 0.81 0.69 0.73 0.77 0.77 0.65 0.81 0.77]
Lowest 0.577 | highest 0.923 | mean 0.760


In [ ]:
# The lesson's five fold scores: mean and spread
fold_scores = np.array([0.88, 0.91, 0.86, 0.90, 0.85])
print("Mean:", fold_scores.mean().round(3))
print("Standard deviation (np.std, as scikit-learn reports):", fold_scores.std().round(4))
print("Sample standard deviation (ddof=1):                 ", fold_scores.std(ddof=1).round(4))

Mean: 0.88
Standard deviation (np.std, as scikit-learn reports): 0.0228
Sample standard deviation (ddof=1):                  0.0255


In [ ]:
from sklearn.model_selection import KFold, cross_val_score

# 5-fold cross-validation: every student is validated once and trained on four times
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")
print("Fold scores:", scores.round(3))
print(f"Mean accuracy {scores.mean():.3f} | standard deviation {scores.std():.3f}")
print("Validation fold sizes:", [len(v) for _, v in kfold.split(X)])

Fold scores: [0.846 0.692 0.76  0.64  0.92 ]
Mean accuracy 0.772 | standard deviation 0.101
Validation fold sizes: [26, 26, 25, 25, 25]


In [ ]:
from sklearn.model_selection import StratifiedKFold

# 37 of the 127 students fail (29%). How many failing students land in each validation fold?
print(f"Failing overall: {(y == 0).sum()} of {len(y)} ({(y == 0).mean():.0%})")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, splitter in [("KFold", kfold), ("Stratified", skf)]:
    fails = [int((y.iloc[v] == 0).sum()) for _, v in splitter.split(X, y)]
    print(f"{name:<10} failing per fold {fails}")

Failing overall: 37 of 127 (29%)
KFold      failing per fold [5, 7, 10, 8, 7]
Stratified failing per fold [8, 8, 7, 7, 7]


In [ ]:
from sklearn.metrics import make_scorer, f1_score

# F1 for fail (the class we want to catch) per fold, with plain and stratified 5-fold
fail_f1 = make_scorer(f1_score, pos_label=0)
for name, splitter in [("KFold", kfold), ("Stratified", skf)]:
    s = cross_val_score(model, X, y, cv=splitter, scoring=fail_f1)
    print(f"{name:<10} {s.round(2)} mean {s.mean():.3f} std {s.std():.3f}")

KFold      [0.6  0.5  0.62 0.31 0.86] mean 0.578 std 0.179
Stratified [0.59 0.67 0.36 0.73 0.75] mean 0.619 std 0.140


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Preprocessing leakage: a scaler fitted on all 127 students before cross-validation has seen every validation fold
knn = KNeighborsClassifier()
leaky = cross_val_score(knn, StandardScaler().fit_transform(X), y, cv=skf).mean()
inside = cross_val_score(Pipeline([("scaler", StandardScaler()), ("classifier", knn)]), X, y, cv=skf).mean()
print(f"Scaler fitted before CV:  {leaky:.3f}")
print(f"Scaler inside the pipeline: {inside:.3f}")

Scaler fitted before CV:  0.771
Scaler inside the pipeline: 0.795


## 2. Stratified and repeated K-fold

In [ ]:
# Stratified 5-fold on our 127 students: what each validation fold holds
print(f"All 127: {(y == 1).sum()} pass, {(y == 0).sum()} fail ({(y == 0).mean():.0%} fail)")
for k, (_, val) in enumerate(skf.split(X, y), 1):
    print(f"Fold {k}: {len(val)} students, {(y.iloc[val] == 1).sum()} pass, {(y.iloc[val] == 0).sum()} fail "
          f"({(y.iloc[val] == 0).mean():.0%})")

All 127: 90 pass, 37 fail (29% fail)
Fold 1: 26 students, 18 pass, 8 fail (31%)
Fold 2: 26 students, 18 pass, 8 fail (31%)
Fold 3: 25 students, 18 pass, 7 fail (28%)
Fold 4: 25 students, 18 pass, 7 fail (28%)
Fold 5: 25 students, 18 pass, 7 fail (28%)


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

# One stratified 5-fold run depends on how the students were shuffled: ten shuffles, ten different estimates
one_run = [cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed), scoring=fail_f1).mean()
           for seed in range(10)]
print("Mean F1 for fail, ten single 5-fold runs:")
print(np.round(one_run[:5], 3))
print(np.round(one_run[5:], 3))
print(f"Range {min(one_run):.3f} to {max(one_run):.3f}")

# Repeated stratified 5-fold: 5 folds x 3 repetitions = 15 fits, averaged
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scores = cross_val_score(model, X, y, cv=rskf, scoring=fail_f1)
print(f"RepeatedStratifiedKFold: {len(scores)} evaluations, mean {scores.mean():.3f}, std {scores.std():.3f}")

Mean F1 for fail, ten single 5-fold runs:
[0.605 0.542 0.556 0.615 0.607]
[0.582 0.587 0.568 0.538 0.559]
Range 0.538 to 0.615
RepeatedStratifiedKFold: 15 evaluations, mean 0.591, std 0.199


In [ ]:
# 3, 5 and 10 folds, each repeated 10 times (so the comparison isn't one lucky shuffle). Cost = number of fits
print("folds  train  failing/fold  mean F1  std    fits")
for k in [3, 5, 10]:
    cv_k = RepeatedStratifiedKFold(n_splits=k, n_repeats=10, random_state=42)
    s = cross_val_score(model, X, y, cv=cv_k, scoring=fail_f1)
    fails = sorted({int((y.iloc[v] == 0).sum()) for _, v in cv_k.split(X, y)})
    print(f"{k:>5}  {1 - 1 / k:>5.0%}  {fails[0]:>6}-{fails[-1]:<5}  {s.mean():>7.3f}  {s.std():.3f}  {len(s):>4}")

folds  train  failing/fold  mean F1  std    fits


    3    67%      12-13       0.582  0.138    30


    5    80%       7-8        0.592  0.162    50


   10    90%       3-4        0.575  0.217   100


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import precision_score, recall_score

# Several metrics in one cross-validation run. "fail" is our positive class, so precision, recall and F1 say pos_label=0;
# ROC-AUC and accuracy don't depend on which class is positive
scoring = {"accuracy": "accuracy", "roc_auc": "roc_auc",
           "precision_fail": make_scorer(precision_score, pos_label=0),
           "recall_fail": make_scorer(recall_score, pos_label=0), "f1_fail": fail_f1}
results = cross_validate(model, X, y, cv=skf, scoring=scoring, return_train_score=True)
print(f"{'metric':<15}{'validation':>11}{'training':>10}")
for name in scoring:
    print(f"{name:<15}{results['test_' + name].mean():>11.3f}{results['train_' + name].mean():>10.3f}")

metric          validation  training
accuracy             0.796     0.789
roc_auc              0.835     0.866
precision_fail       0.687     0.666
recall_fail          0.593     0.554
f1_fail              0.619     0.604
